## Objective & Tasks:

Overview: As an automotive supplier, we are interested in understanding the broader trends in the automotive industry.

Data Transformation: Transform the data into a format suitable for further analysis. Justify the choices you make during this process.

**Step 1: Clean numeric data (replace invalid/placeholder values with NULL) → ensures meaningful analysis.**

**Step 2: Move id first → improves readability.**

**Step 3: Identify fully NULL columns → remove or ignore empty data.**

**Step 4: Write to Delta with mergeSchema for schema enforcement and evolution**

## Outcome:
The gold table has cleaned numeric data, id first, no fully NULL columns, ready for analysis.

## Step 1: Numeric data cleaning and validation

- Marks “not available” scores as NULL
- Converts meaningless zeros in EV/PHEV fields to NULL
- Ensures core physical numeric fields (engine size, CO2, fuel consumption) are non-negative
- Prepares the dataset for reliable reporting, dashboards, and analytics

In [0]:
from pyspark.sql.functions import col, when

# Load silver table
df_gold = spark.read.table("hive_metastore.epa_vehicle.silver")

# --- Scores where -1 = "not available"
scores_cols = ["feScore", "ghgScore", "ghgScoreA"]
for c in scores_cols:
    df_gold = df_gold.withColumn(c, when(col(c) >= 0, col(c)))

# --- Optional EV / PHEV numeric fields: convert 0 → NULL if not meaningful
ev_cols = [
    "range", "rangeCity", "rangeCityA", "rangeHwy", "rangeHwyA",
    "rangeA", "charge120", "charge240", "UCity", "UCityA",
    "UHighway", "UHighwayA"
]
for c in ev_cols:
    df_gold = df_gold.withColumn(c, when(col(c) > 0, col(c)))

# --- Core physical numeric fields: ensure valid values only
core_numeric = ["displ", "co2", "barrels08"]
for c in core_numeric:
    df_gold = df_gold.withColumn(c, when(col(c) >= 0, col(c)))

# Optional: check result
df_gold.select(
    ["id"] + scores_cols + ev_cols + core_numeric
).show(5, False)


## Step 2: Move id column first

Moving the column first since it's the primary key and to better visualize it.

In [0]:
# Get current columns
cols = df_gold.columns

# Reorder: put 'id' first, keep the rest in order
new_order = ["id"] + [c for c in cols if c != "id"]

# Apply new order
df_silver = df_gold.select(new_order)

# Validate
display(df_silver)


## Step 3: This will give a list of columns where every row is NULL.

Result:
None of the columns are fully Null.
We can't drop any column since it may contain valid data.

In [0]:
from pyspark.sql.functions import col, sum

# List columns where all values are null
all_null_cols = [c for c in df_silver.columns 
                 if df_silver.select(sum(col(c).isNotNull().cast("int"))).collect()[0][0] == 0]

print("Columns that are completely null:", all_null_cols)


## Step 4: Write to Delta with mergeSchema for schema enforcement and evolution

In [0]:
# Define the gold path and table
gold_path = "dbfs:/mnt/epa/gold"
gold_table = "epa_vehicle.gold"

# Write df_gold_clean to Delta format
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_path)

print(f"Gold table saved at {gold_path}")

# Create table if not exists
spark.sql(f"CREATE TABLE IF NOT EXISTS {gold_table} USING DELTA LOCATION '{gold_path}'")

# Verify the table
display(spark.sql(f"SELECT * FROM {gold_table} LIMIT 10"))